# Generating training data

Once you have annotations, now you want to make training data

For now this generation of training data will be weird, since we will be using data from two sources: my old 3D gui data (originally sent to Laetitia) and the new GUI data

Format of training data:

stacks of frames cropped down to the ROI
The bps in this cropped coord system in a .csv file

# Imports

In [2]:
import numpy as np
import h5py
import cv2
import pandas as pd
import csv
from joblib import load
import matplotlib.pyplot as plt
%matplotlib widget
import time
%load_ext autoreload
import os
%autoreload 2

# Our custom-made imports
import sys
sys.path.append('./lib/')
from DataManagement import SplitdataManager, Filepaths, Calibration

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Loading Data

## Params for this experiment

In [3]:
# set an experiment folder path and make a splitdata manager to get at frames
experiment_path = '/media/stephens-group/guest_drive/labelling/complete_shortfin_experiment/FishTank20200130_153857/'
splitman = SplitdataManager(experiment_path)


# set the ROIS
# bbs_info = {'xz_tlc': [175, 175],
#             'xz_width': 930,
#             'xz_height': 820,
#             'xy_tlc': [175, 175],
#             'xy_width': 930,
#             'xy_height': 820,
#             'yz_tlc': [175, 175],
#             'yz_width': 930,
#             'yz_height': 820}


bbs_info = {'xz_tlc': [175, 175],
             'xz_width': 930,
             'xz_height': 820,
             'xy_tlc': [246, 120],
             'xy_width': 760,
             'xy_height': 755,
             'yz_tlc': [180, 180],
             'yz_width': 940,
             'yz_height': 800}

# get the crop shapes in numpy (not opencv) form
xz_crop_shape = (bbs_info['xz_height'], bbs_info['xz_width'])
xy_crop_shape = (bbs_info['xy_height'], bbs_info['xy_width'])
yz_crop_shape = (bbs_info['yz_height'], bbs_info['yz_width'])

In [4]:
xz_crop_shape

(820, 930)

In [5]:
xy_crop_shape

(755, 760)

In [6]:
yz_crop_shape

(800, 940)

## new data

In [7]:
# save the final results
folderPath = '/home/liam/Data/returned_labels/'
combinedPath = os.path.join(folderPath, 'combined_annotations.h5')
with h5py.File(combinedPath, 'r') as hf:
    annotated_frames = hf['annotated_frames'][:]
    missing_bps = hf['missing_bps'][:]
    tracks_3D = hf['tracks_3D'][:]
    tracks_imCoords = hf['tracks_imCoords'][:]
    
# how many annotated frames
numAnnotated = annotated_frames[annotated_frames==True].shape[0]
numAnnotated

# get an array of annotated frame numbers
annotated_fIdxs = np.where(annotated_frames==True)[0]


In [8]:
tracks_imCoords.shape

(3, 501943, 2, 3, 2)

# Functions

In [9]:
def extract_frames_for_global_frame_idx(splitDataMan, global_index):
    ''' Given a SplitDataManager containing all of the path information,
        and a global frame index for the experiment,
        return the frames
    '''
    # get the moviePaths and local frame index
    sd_fld_paths, local_index = splitDataMan.return_splitdata_folder_and_local_idx_for_global_frameIdx(global_index)
    movPaths = [sd_fld_paths[i]+'.mp4' for i in range(3)]
    
    # grab the frames
    caps = [cv2.VideoCapture(video_path) for video_path in movPaths]          # open the movie caps
    [cap.set(1, local_index) for cap in caps]                                 # go to current frame
    # grab the frame from each camera
    if caps[0].isOpened() and caps[1].isOpened() and caps[2].isOpened():  
        # grab the 3 images
        ret_xz, frame_xz = caps[0].read()
        frame_xz = cv2.cvtColor(frame_xz, cv2.COLOR_BGR2GRAY)
        if not ret_xz:
            raise TypeError('xz movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
        ret_xy, frame_xy = caps[1].read()
        frame_xy = cv2.cvtColor(frame_xy, cv2.COLOR_BGR2GRAY)
        if not ret_xy:
            raise TypeError('xy movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
        ret_yz, frame_yz = caps[2].read()
        frame_yz = cv2.cvtColor(frame_yz, cv2.COLOR_BGR2GRAY)
        if not ret_yz:
            raise TypeError('yz movie for splitdata_idx={0} not opening'.format(splitdata_idx.value))
        frames = [frame_xz, frame_xy, frame_yz]
    # close everything before we finish
    [caps[capIdx].release() for capIdx in range(3)]
    
    return frames


def crop_frames(frames, bboxs_info):
    ''' Given the list of frames, one for each camera,
        use the bbs_info to return cropped versions of the frames
    '''
    for camIdx, camName in enumerate(['xz', 'xy', 'yz']):
        frame = np.copy(frames[camIdx])
        x, y = bboxs_info[camName + '_tlc']
        w = bboxs_info[camName + '_width']
        h = bboxs_info[camName + '_height']
        # we swap x,y for numpy row,col indexing
        frame = frame[y:y+h, x:x+w]
        frames[camIdx] = frame
    return frames



def write_labels_csv_files(tracks_imCoords, annotated_fIdxs, saveFolder, bb_info):
    ''' Write the labels_*.csv files from the FishTank_GUI information
    
    -- Inputs --
    tracks_imCoords: array of shape (numCams, numFramesInExperiment, numFish, numBodyPoints, 2)
                     The image coordinates from the GUI
    annotated_fIdxs: 1D array containing the fIdxs of annotated frames
    saveFolder:      the folderPath to save the labels files in
    
    -- Returns --
    None -  but files are written.
    '''
    # get the missing_bps and labels arrays
    labels_data = np.copy(tracks_imCoords[:, annotated_fIdxs])
    labels_data = np.flip(labels_data, axis=-1) # Swap imcoords from opencv to numpy
    missing_bps_data = np.copy(missing_bps[:, annotated_fIdxs])
    
    # now you need to account for the cropping in the labels
    labels_data_cropped = np.copy(labels_data)
    for i,camName in enumerate(['xz', 'xy', 'yz']):
        # get the tlc for this cam, swapped for numpy indexing
        tlc = bb_info[camName+'_tlc'][::-1]
        labels_data_cropped[i] = labels_data[i] - tlc
        
    # define the savepaths
    xzLabelsPath = os.path.join(saveFolder, 'labels_xz.csv')
    xyLabelsPath = os.path.join(saveFolder, 'labels_xy.csv')
    yzLabelsPath = os.path.join(saveFolder, 'labels_yz.csv')
    csv_paths = [xzLabelsPath, xyLabelsPath, yzLabelsPath]
    
    # derive some shapes
    numCams, _, numFish, numBodyPoints, _ = labels_data_cropped.shape
    numAnnotated = annotated_fIdxs.shape[0]
    
    # write the csv files
    for camIdx in range(numCams):
    
        with open(csv_paths[camIdx], "w") as results_csv_f:
            writer = csv.writer(results_csv_f)

            for i in range(numAnnotated): 
                for fishIdx in range(numFish):
                    for bpIdx in range(numBodyPoints):
                        visible = ~np.copy(missing_bps[camIdx, i, fishIdx, bpIdx])
                        x = labels_data_cropped[camIdx, i, fishIdx, bpIdx, 0]
                        y = labels_data_cropped[camIdx, i, fishIdx, bpIdx, 1]
                        pointInfo = [i, fishIdx, bpIdx, x, y, visible]
                        writer.writerow(pointInfo)
        

# Prepare the new GUI data

In [10]:
# define the folder to save the results from the new GUI
newDataSaveFolder =  '/home/liam/Data/returned_labels/new_training_data/just_new'


In [11]:
#---- Frames ----- #
t0 = time.time()

# frame paths
xzFramesPath = os.path.join(newDataSaveFolder, 'frames_xz.h5')
xyFramesPath = os.path.join(newDataSaveFolder, 'frames_xy.h5')
yzFramesPath = os.path.join(newDataSaveFolder, 'frames_yz.h5')

# preallocate output
xz_cropped_frames = np.zeros((numAnnotated, 1, xz_crop_shape[0], xz_crop_shape[1]), dtype=np.uint8)
xy_cropped_frames = np.zeros((numAnnotated, 1, xy_crop_shape[0], xy_crop_shape[1]), dtype=np.uint8)
yz_cropped_frames = np.zeros((numAnnotated, 1, yz_crop_shape[0], yz_crop_shape[1]), dtype=np.uint8)

# gather the frames
for i,fIdx in enumerate(annotated_fIdxs):
    fIdx_frames = extract_frames_for_global_frame_idx(splitman, fIdx)
    fIdx_frames_cropped = crop_frames(fIdx_frames, bbs_info)
    # record
    xz_cropped_frames[i, 0] = fIdx_frames_cropped[0]
    xy_cropped_frames[i, 0] = fIdx_frames_cropped[1]
    yz_cropped_frames[i, 0] = fIdx_frames_cropped[2]
    
# save
with h5py.File(xzFramesPath, 'w') as hf:
    hf.create_dataset('frames', data=xz_cropped_frames)
with h5py.File(xyFramesPath, 'w') as hf:
    hf.create_dataset('frames', data=xy_cropped_frames)
with h5py.File(yzFramesPath, 'w') as hf:
    hf.create_dataset('frames', data=yz_cropped_frames)
    
print(time.time()-t0)

305.88508558273315


In [12]:
# ---- Labels -------------#

In [13]:
write_labels_csv_files(tracks_imCoords, annotated_fIdxs, newDataSaveFolder, bbs_info)

# Prepare the old GUI data

In this section of code we want to extract the data I shared with laetitia for her first training, 
in the current format we desire. By this I mean, make this data match the data from the new GUI

-- inputs --
fish_annotations_liam.h5 and fish_annotations_tatsuo.h5
Two h5 files that contain several datasets, frames_, labels_ and missing_,
where _ is each of the cameras

In [14]:
def write_labels_csv_files_old_data(labels_xz, labels_xy, labels_yz, missing_xz, missing_xy, missing_yz, saveFolder, bb_info):
    ''' Write the labels_*.csv files from the FishTank_GUI information
    
    -- Inputs --
    labels_** : the labels arrays from the old data
    saveFolder:      the folderPath to save the labels files in
    
    -- Returns --
    None -  but files are written.
    '''
    # swap to numpy format from opencv
    labels_xz =  np.flip(labels_xz, axis=-1)
    labels_xy =  np.flip(labels_xy, axis=-1)
    labels_yz =  np.flip(labels_yz, axis=-1)
    
    # now you need to account for the cropping in the labels
    labels_xz_cropped = np.copy(labels_xz)
    tlc = bb_info['xz_tlc'][::-1] #swapped for numpy indexing
    labels_xz_cropped = labels_xz - tlc
    
    labels_xy_cropped = np.copy(labels_xy)
    tlc = bb_info['xy_tlc'][::-1] #swapped for numpy indexing
    labels_xy_cropped = labels_xy - tlc
    
    labels_yz_cropped = np.copy(labels_yz)
    tlc = bb_info['yz_tlc'][::-1] #swapped for numpy indexing
    labels_yz_cropped = labels_yz - tlc
    
    labels_cropped = [labels_xz_cropped, labels_xy_cropped, labels_yz_cropped]
        
    # define the savepaths
    xzLabelsPath = os.path.join(saveFolder, 'labels_xz.csv')
    xyLabelsPath = os.path.join(saveFolder, 'labels_xy.csv')
    yzLabelsPath = os.path.join(saveFolder, 'labels_yz.csv')
    csv_paths = [xzLabelsPath, xyLabelsPath, yzLabelsPath]
    
    # wrap up the missing arrays for ease
    missing_bps = [missing_xz, missing_xy, missing_yz]
    
    # derive some shapes
    _, numFish, numBodyPoints, _ = labels_xz.shape
    numCams = 3
    numAnnotated = labels_xz.shape[0]
    
    # write the csv files
    for camIdx in range(numCams):
    
        with open(csv_paths[camIdx], "w") as results_csv_f:
            writer = csv.writer(results_csv_f)

            for i in range(numAnnotated): 
                for fishIdx in range(numFish):
                    for bpIdx in range(numBodyPoints):
                        if missing_bps[camIdx][i, fishIdx, bpIdx] == 0:
                            visible = True
                        else:
                            visible = False
                        x = labels_cropped[camIdx][ i, fishIdx, bpIdx, 0]
                        y = labels_cropped[camIdx][ i, fishIdx, bpIdx, 1]
                        pointInfo = [i, fishIdx, bpIdx, x, y, visible]
                        writer.writerow(pointInfo)

## Liam

In [15]:
# define the folder to save the results from the new GUI
oldDataSaveFolder =  '/home/liam/Data/returned_labels/new_training_data/just_old_liam/'

# --- load the data ---- #
old_folder_path = '/home/liam/Data/returned_labels/old_labels/what_I_provided/fish_annotations'
liam_path = os.path.join(old_folder_path, 'fish_annotations_liam.h5')

with h5py.File(liam_path, 'r') as hf:
    frames_xz = hf['frames_xz'][:]
    frames_xy = hf['frames_xy'][:]
    frames_yz = hf['frames_yz'][:]
    
    labels_xz = hf['labels_xz'][:]
    labels_xy = hf['labels_xy'][:]
    labels_yz = hf['labels_yz'][:]
    
    missing_xz = hf['missing_xz'][:]
    missing_xy = hf['missing_xy'][:]
    missing_yz = hf['missing_yz'][:]
    
    
oldnumAnnotated = frames_xz.shape[0]
    
    
# --- define the save paths ---- #

# frame paths
xzFramesPath = os.path.join(oldDataSaveFolder, 'frames_xz.h5')
xyFramesPath = os.path.join(oldDataSaveFolder, 'frames_xy.h5')
yzFramesPath = os.path.join(oldDataSaveFolder, 'frames_yz.h5')

# label paths
xzLabelsPath = os.path.join(oldDataSaveFolder, 'labels_xz.csv')
xyLabelsPath = os.path.join(oldDataSaveFolder, 'labels_xy.csv')
yzLabelsPath = os.path.join(oldDataSaveFolder, 'labels_yz.csv')
csv_paths = [xzLabelsPath, xyLabelsPath, yzLabelsPath]


#---- Frames ----- #
t0 = time.time()

# preallocate output
xz_cropped_frames = np.zeros((oldnumAnnotated, 1, xz_crop_shape[0], xz_crop_shape[1]), dtype=np.uint8)
xy_cropped_frames = np.zeros((oldnumAnnotated, 1, xy_crop_shape[0], xy_crop_shape[1]), dtype=np.uint8)
yz_cropped_frames = np.zeros((oldnumAnnotated, 1, yz_crop_shape[0], yz_crop_shape[1]), dtype=np.uint8)

# gather the frames
for i in range(frames_xz.shape[0]):
    fIdx_xzframe = frames_xz[i]
    fIdx_xyframe = frames_xy[i]
    fIdx_yzframe = frames_yz[i]
    fIdx_frames = [fIdx_xzframe, fIdx_xyframe, fIdx_yzframe]
    fIdx_frames_cropped = crop_frames(fIdx_frames, bbs_info)
    # record
    xz_cropped_frames[i, 0] = fIdx_frames_cropped[0]
    xy_cropped_frames[i, 0] = fIdx_frames_cropped[1]
    yz_cropped_frames[i, 0] = fIdx_frames_cropped[2]
    
# save
with h5py.File(xzFramesPath, 'w') as hf:
    hf.create_dataset('frames', data=xz_cropped_frames)
with h5py.File(xyFramesPath, 'w') as hf:
    hf.create_dataset('frames', data=xy_cropped_frames)
with h5py.File(yzFramesPath, 'w') as hf:
    hf.create_dataset('frames', data=yz_cropped_frames)
    
print(time.time()-t0)


# ----- Labels ------- #
write_labels_csv_files_old_data(labels_xz, labels_xy, labels_yz, missing_xz, missing_xy, missing_yz, oldDataSaveFolder, bbs_info)

2.304443359375


## tatsuo

In [16]:
# define the folder to save the results from the new GUI
oldDataSaveFolder =  '/home/liam/Data/returned_labels/new_training_data/just_old_tatsuo/'

# --- load the data ---- #
old_folder_path = '/home/liam/Data/returned_labels/old_labels/what_I_provided/fish_annotations'
tatsuo_path = os.path.join(old_folder_path, 'fish_annotations_tatsuo.h5')

with h5py.File(tatsuo_path, 'r') as hf:
    frames_xz = hf['frames_xz'][:]
    frames_xy = hf['frames_xy'][:]
    frames_yz = hf['frames_yz'][:]
    
    labels_xz = hf['labels_xz'][:]
    labels_xy = hf['labels_xy'][:]
    labels_yz = hf['labels_yz'][:]
    
    missing_xz = hf['missing_xz'][:]
    missing_xy = hf['missing_xy'][:]
    missing_yz = hf['missing_yz'][:]
    
oldnumAnnotated = frames_xz.shape[0]
    
# --- define the save paths ---- #

# frame paths
xzFramesPath = os.path.join(oldDataSaveFolder, 'frames_xz.h5')
xyFramesPath = os.path.join(oldDataSaveFolder, 'frames_xy.h5')
yzFramesPath = os.path.join(oldDataSaveFolder, 'frames_yz.h5')

# label paths
xzLabelsPath = os.path.join(oldDataSaveFolder, 'labels_xz.csv')
xyLabelsPath = os.path.join(oldDataSaveFolder, 'labels_xy.csv')
yzLabelsPath = os.path.join(oldDataSaveFolder, 'labels_yz.csv')
csv_paths = [xzLabelsPath, xyLabelsPath, yzLabelsPath]


#---- Frames ----- #
t0 = time.time()

# preallocate output
xz_cropped_frames = np.zeros((oldnumAnnotated, 1, xz_crop_shape[0], xz_crop_shape[1]), dtype=np.uint8)
xy_cropped_frames = np.zeros((oldnumAnnotated, 1, xy_crop_shape[0], xy_crop_shape[1]), dtype=np.uint8)
yz_cropped_frames = np.zeros((oldnumAnnotated, 1, yz_crop_shape[0], yz_crop_shape[1]), dtype=np.uint8)

# gather the frames
for i in range(frames_xz.shape[0]):
    fIdx_xzframe = frames_xz[i]
    fIdx_xyframe = frames_xy[i]
    fIdx_yzframe = frames_yz[i]
    fIdx_frames = [fIdx_xzframe, fIdx_xyframe, fIdx_yzframe]
    fIdx_frames_cropped = crop_frames(fIdx_frames, bbs_info)
    # record
    xz_cropped_frames[i, 0] = fIdx_frames_cropped[0]
    xy_cropped_frames[i, 0] = fIdx_frames_cropped[1]
    yz_cropped_frames[i, 0] = fIdx_frames_cropped[2]
    
# save
with h5py.File(xzFramesPath, 'w') as hf:
    hf.create_dataset('frames', data=xz_cropped_frames)
with h5py.File(xyFramesPath, 'w') as hf:
    hf.create_dataset('frames', data=xy_cropped_frames)
with h5py.File(yzFramesPath, 'w') as hf:
    hf.create_dataset('frames', data=yz_cropped_frames)
    
print(time.time()-t0)


# ----- Labels ------- #
write_labels_csv_files_old_data(labels_xz, labels_xy, labels_yz, missing_xz, missing_xy, missing_yz, oldDataSaveFolder, bbs_info)

1.6473238468170166


# Combining the data

Combine several training data folders into one master folder

In [17]:
# set the load folder paths
folder1 = '/home/liam/Data/returned_labels/new_training_data/just_new'
folder2 = '/home/liam/Data/returned_labels/new_training_data/just_old_liam/'
folder3 = '/home/liam/Data/returned_labels/new_training_data/just_old_tatsuo/'
folders = [folder1, folder2, folder3]

# create the combined folder path
combined_folder = '/home/liam/Data/returned_labels/new_training_data/dataset'

In [18]:
# load, combine, save

# load
numFrames_list = []
frames_xz_list = []
frames_xy_list = []
frames_yz_list = []
for load_folder in folders:
    with h5py.File(os.path.join(load_folder, 'frames_xz.h5'), 'r') as hf:
        frames_xz = hf['frames'][:]
        frames_xz_list.append(frames_xz)
    with h5py.File(os.path.join(load_folder, 'frames_xy.h5'), 'r') as hf:
        frames_xy = hf['frames'][:]
        frames_xy_list.append(frames_xy)
    with h5py.File(os.path.join(load_folder, 'frames_yz.h5'), 'r') as hf:
        frames_yz = hf['frames'][:] 
        frames_yz_list.append(frames_yz)
    nfs = frames_xz.shape[0]
    numFrames_list.append(nfs)
        
# combine 
frames_xz_combined = np.concatenate(frames_xz_list, axis=0)
frames_xy_combined = np.concatenate(frames_xy_list, axis=0)
frames_yz_combined = np.concatenate(frames_yz_list, axis=0)

# save
with h5py.File(os.path.join(combined_folder, 'frames_xz.h5'), 'w') as hf:
    hf.create_dataset('frames', data=frames_xz_combined)
with h5py.File(os.path.join(combined_folder, 'frames_xy.h5'), 'w') as hf:
    hf.create_dataset('frames', data=frames_xy_combined)
with h5py.File(os.path.join(combined_folder, 'frames_yz.h5'), 'w') as hf:
    hf.create_dataset('frames', data=frames_yz_combined)

In [19]:
numFrames_list

[592, 209, 162]

In [20]:
# get the offset for each dataset for concatenating
start_idxs = [0]
for i in range(1, len(numFrames_list)):
    start_idx = start_idxs[i-1] + numFrames_list[i-1]
    start_idxs.append(start_idx)

In [21]:
start_idxs

[0, 592, 801]

In [22]:
# labels 

labels_df_xz_list = []
labels_df_xy_list = []
labels_df_yz_list = []

# gather up the labels, accounting for the offset in frame number from stacking
for i,load_folder in enumerate(folders):
    
    frame_idx_offset = start_idxs[i]
    
    labels_path_xz = os.path.join(load_folder, 'labels_xz.csv')
    labels_path_xy = os.path.join(load_folder, 'labels_xy.csv')
    labels_path_yz = os.path.join(load_folder, 'labels_yz.csv')
    
    labels_df_xz = pd.read_csv(labels_path_xz, names=["frame_index", "instance_index", "joint_index", "x", "y", "visible"])
    labels_df_xz["frame_index"] = labels_df_xz["frame_index"] + frame_idx_offset
    
    labels_df_xy = pd.read_csv(labels_path_xy, names=["frame_index", "instance_index", "joint_index", "x", "y", "visible"])
    labels_df_xy["frame_index"] = labels_df_xy["frame_index"] + frame_idx_offset
    
    labels_df_yz = pd.read_csv(labels_path_yz, names=["frame_index", "instance_index", "joint_index", "x", "y", "visible"])
    labels_df_yz["frame_index"] = labels_df_yz["frame_index"] + frame_idx_offset
    
    labels_df_xz_list.append(labels_df_xz)
    labels_df_xy_list.append(labels_df_xy)
    labels_df_yz_list.append(labels_df_yz)
    
    
# concatenate the arrays
labels_xz_combined = pd.concat(labels_df_xz_list)
labels_xy_combined = pd.concat(labels_df_xy_list)
labels_yz_combined = pd.concat(labels_df_yz_list)

## TEMP ##
# set all visible to true
labels_xz_combined["visible"] = True
labels_xy_combined["visible"] = True
labels_yz_combined["visible"] = True

# save
labels_xz_combined.to_csv(os.path.join(combined_folder, 'labels_xz.csv'), header = False, index=False)
labels_xy_combined.to_csv(os.path.join(combined_folder, 'labels_xy.csv'), header = False, index=False)
labels_yz_combined.to_csv(os.path.join(combined_folder, 'labels_yz.csv'), header = False, index=False)

# Why is there a difference between XY and XZ&YZ?

In [ ]:
# One of the new things is the 'visible' array. Is there a difference there?

In [52]:
labels_xz_combined.head()

,frame_index,instance_index,joint_index,x,y,visible
0,0,0,0,673.350102,777.289398,True
1,0,0,1,672.891314,781.690106,True
2,0,0,2,670.493800,796.978575,True
3,0,1,0,666.950279,688.482104,True
4,0,1,1,670.554717,693.567408,True


In [74]:
bps_vis = []

for frame_index, cur_label in labels_xz_combined.groupby("frame_index"):
    for inst_index, instance_data in cur_label.groupby("instance_index"):
        bps_visible = instance_data['visible'].values
        if np.all(bps_visible==False):
            bps_vis.append(0)
        else:
            bps_vis.append(1)
            
bps_vis = np.array(bps_vis)

bps_vis[bps_vis==0]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0])

In [75]:
bps_vis = []

for frame_index, cur_label in labels_xy_combined.groupby("frame_index"):
    for inst_index, instance_data in cur_label.groupby("instance_index"):
        bps_visible = instance_data['visible'].values
        if np.all(bps_visible==False):
            bps_vis.append(0)
        else:
            bps_vis.append(1)
            
bps_vis = np.array(bps_vis)

bps_vis[bps_vis==0]

array([], dtype=int64)

In [76]:
bps_vis = []

for frame_index, cur_label in labels_yz_combined.groupby("frame_index"):
    for inst_index, instance_data in cur_label.groupby("instance_index"):
        bps_visible = instance_data['visible'].values
        if np.all(bps_visible==False):
            bps_vis.append(0)
        else:
            bps_vis.append(1)
            
bps_vis = np.array(bps_vis)

bps_vis[bps_vis==0]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0])

In [ ]:
THIS DOES SEEM TO PERHAPS BE THE PROBLEM

In [42]:
frames_xz_combined.dtype

dtype('uint8')

In [43]:
frames_xy_combined.dtype

dtype('uint8')

In [39]:
plt.figure()
plt.imshow(frames_xz_combined[-1, 0])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [40]:
plt.figure()
plt.imshow(frames_xy_combined[-1, 0])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [41]:
plt.figure()
plt.imshow(frames_yz_combined[-1, 0])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [46]:
xz_lbs = labels_df_xz[['x', 'y']].values
xz_lbs

array([[673.35010192, 777.28939794],
       [672.89131356, 781.69010597],
       [670.49380003, 796.97857464],
       ...,
       [614.23570829, 517.50867742],
       [618.44404351, 512.19368453],
       [627.24452189, 499.77865637]])

In [47]:
np.where(np.isnan(xz_lbs))

(array([], dtype=int64), array([], dtype=int64))

In [48]:
xy_lbs = labels_df_xy[['x', 'y']].values
np.where(np.isnan(xy_lbs))

(array([], dtype=int64), array([], dtype=int64))

In [49]:
yz_lbs = labels_df_yz[['x', 'y']].values
np.where(np.isnan(yz_lbs))

(array([], dtype=int64), array([], dtype=int64))

In [29]:
labels_df_xz = pd.read_csv(os.path.join(combined_folder, 'labels_xz.csv'), names=["frame_index", "instance_index", "joint_index", "x", "y", "visible"])
labels_df_xz

,frame_index,instance_index,joint_index,x,y,visible
0,0,0,0,673.350102,777.289398,True
1,0,0,1,672.891314,781.690106,True
2,0,0,2,670.493800,796.978575,True
3,0,1,0,666.950279,688.482104,True
4,0,1,1,670.554717,693.567408,True
...,...,...,...,...,...,...
5773,962,0,1,605.108101,483.931349,True
5774,962,0,2,587.983728,453.327561,True
5775,962,1,0,614.235708,517.508677,True
5776,962,1,1,618.444044,512.193685,False


In [30]:
labels_df_xy = pd.read_csv(os.path.join(combined_folder, 'labels_xy.csv'), names=["frame_index", "instance_index", "joint_index", "x", "y", "visible"])
labels_df_xy

,frame_index,instance_index,joint_index,x,y,visible
0,0,0,0,542.910000,113.080000,True
1,0,0,1,525.692202,110.926193,True
2,0,0,2,492.477497,101.113656,True
3,0,1,0,623.950000,183.040000,True
4,0,1,1,606.060000,179.940000,True
...,...,...,...,...,...,...
5773,962,0,1,258.640368,365.017124,True
5774,962,0,2,275.867625,389.107325,True
5775,962,1,0,265.527843,338.624196,True
5776,962,1,1,277.565335,342.604230,True


In [31]:
labels_df_yz = pd.read_csv(os.path.join(combined_folder, 'labels_yz.csv'), names=["frame_index", "instance_index", "joint_index", "x", "y", "visible"])
labels_df_yz

,frame_index,instance_index,joint_index,x,y,visible
0,0,0,0,661.913157,275.340216,True
1,0,0,1,658.879049,295.178195,True
2,0,0,2,651.029747,333.269819,True
3,0,1,0,673.962298,178.203566,True
4,0,1,1,674.869413,199.429061,True
...,...,...,...,...,...,...
5773,962,0,1,585.641181,613.841899,True
5774,962,0,2,573.418100,593.835470,True
5775,962,1,0,592.593915,603.906420,True
5776,962,1,1,598.405744,589.804774,True
